# 🎬 Cinematic Engine V17
### شغّل الـ cells بالترتيب: 1 ثم 2 ثم 3
**قبل أي حاجة:** `Runtime → Change runtime type → T4 GPU → Save`


In [ ]:
# ════════════════════════════════════════
# CELL 1 — SETUP
# شغّله مرة واحدة في بداية كل session
# ════════════════════════════════════════
import os, subprocess, sys

# تأكد من GPU
import torch
assert torch.cuda.is_available(), '❌ مفيش GPU — Runtime → Change runtime type → T4 GPU'
print(f'✅ GPU: {torch.cuda.get_device_name(0)} | VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f}GB')

# تحميل المشروع
REPO = 'https://github.com/michelluke1984-cyber/cinematic-engine'
DIR  = '/content/cinematic-engine'
if not os.path.exists(DIR):
    print('⬇️ تحميل المشروع...')
    assert os.system(f'git clone -q {REPO} {DIR}') == 0, '❌ فشل تحميل المشروع'
else:
    os.system(f'cd {DIR} && git pull -q')
print('✅ المشروع جاهز')
os.chdir(DIR)

# تثبيت الـ packages
print('📦 تثبيت الـ packages...')
pkgs = [
    'pip install -q diffusers transformers accelerate xformers einops',
    'pip install -q gfpgan realesrgan nltk sentencepiece tqdm',
    'pip install -q "gradio>=4.0.0" bitsandbytes Pillow',
    'pip install -q insightface onnxruntime-gpu',
    'pip install -q websockets nest_asyncio pyngrok',
]
for p in pkgs:
    subprocess.run(p, shell=True, capture_output=True)
    print('.', end='', flush=True)
print(' ✅')

import nltk
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

# تحميل model weights
print('⬇️ تحميل الـ models...')
for fname, url in [
    ('GFPGANv1.3.pth',           'https://github.com/TencentARC/GFPGAN/releases/download/v1.3.0/GFPGANv1.3.pth'),
    ('realesr-general-x4v3.pth', 'https://github.com/xinntao/Real-ESRGAN/releases/download/v0.2.5.0/realesr-general-x4v3.pth'),
]:
    if not os.path.exists(fname):
        subprocess.run(f'wget -q -O {fname} {url}', shell=True)
    print(f'  ✅ {fname}')

print('\n✅ CELL 1 خلص — شغّل CELL 2')

In [ ]:
# ════════════════════════════════════════
# CELL 2 — تحميل الـ Engine
# ════════════════════════════════════════
import sys, os, importlib.util, torch, builtins

DIR = '/content/cinematic-engine'
assert os.path.exists(DIR), '❌ شغّل CELL 1 الأول'
os.chdir(DIR)

ENGINE = os.path.join(DIR, 'cinematic_engine_v16_pro.py')
assert os.path.exists(ENGINE), f'❌ الملف مش موجود: {ENGINE}'

print('⏳ تحميل CinematicEngineV16...')

# الآن آمن — STEP 7 محمي بـ if __name__ == '__main__'
spec  = importlib.util.spec_from_file_location('cev16', ENGINE)
cev16 = importlib.util.module_from_spec(spec)
sys.modules['cev16'] = cev16
spec.loader.exec_module(cev16)

# تصدير كل الـ classes للـ scope الحالي
for k, v in vars(cev16).items():
    if not k.startswith('_'):
        globals()[k] = v
        sys.modules['__main__'].__dict__[k] = v

assert 'CinematicEngineV16' in globals(), '❌ CinematicEngineV16 مش موجودة'

# إنشاء الـ engine
print('⏳ Initialising engine...')
engine = CinematicEngineV16()

# حفظ في كل الأماكن عشان Cell 3 يلاقيه
globals()['engine'] = engine
sys.modules['__main__'].__dict__['engine'] = engine
builtins._cev17_engine = engine

print(f'✅ Engine جاهز: {type(engine).__name__}')
print(f'✅ GPU: {torch.cuda.get_device_name(0)}')
print('\n✅ CELL 2 خلص — شغّل CELL 3')


In [ ]:
# ════════════════════════════════════════
# CELL 3 — تشغيل الـ Bridge
# هيفضل شغّال — اضغط ⬛ عشان توقف
# ════════════════════════════════════════
import sys, os, asyncio, importlib.util, nest_asyncio, time, builtins, threading
nest_asyncio.apply()

DIR = '/content/cinematic-engine'
assert os.path.exists(DIR), '❌ شغّل CELL 1 الأول'
os.chdir(DIR)

# تحرير البورت 7860
os.system('fuser -k 7860/tcp 2>/dev/null || true')
time.sleep(2)

# جلب الـ engine
_engine = (
    globals().get('engine') or
    sys.modules.get('__main__', {}).__dict__.get('engine') or
    getattr(builtins, '_cev17_engine', None)
)
assert _engine is not None, '❌ Engine مش موجود — شغّل CELL 2 الأول'
print(f'✅ Engine: {type(_engine).__name__}')

# تحميل الـ bridge
BRIDGE = os.path.join(DIR, 'cev17_backend.py')
assert os.path.exists(BRIDGE), f'❌ {BRIDGE} مش موجود'
spec = importlib.util.spec_from_file_location('bridge', BRIDGE)
bridge_mod = importlib.util.module_from_spec(spec)
sys.modules['bridge'] = bridge_mod
spec.loader.exec_module(bridge_mod)
print('✅ Bridge loaded')

# Cloudflare Tunnel — مجاني 100% بدون كارت بنك
import subprocess, re

# تحميل cloudflared
if not os.path.exists('/tmp/cloudflared'):
    print('⬇️ تحميل cloudflared...')
    os.system('wget -q -O /tmp/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64')
    os.system('chmod +x /tmp/cloudflared')
    print('✅ cloudflared جاهز')

# تشغيل الـ tunnel في background
tunnel_url = None
tunnel_ready = threading.Event()

def run_tunnel():
    global tunnel_url
    proc = subprocess.Popen(
        ['/tmp/cloudflared', 'tunnel', '--url', 'http://localhost:7860', '--no-autoupdate'],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True
    )
    for line in proc.stdout:
        if 'trycloudflare.com' in line:
            match = re.search(r'https://[\w-]+\.trycloudflare\.com', line)
            if match:
                tunnel_url = match.group(0)
                tunnel_ready.set()
                break

t = threading.Thread(target=run_tunnel, daemon=True)
t.start()

print('⏳ جاري فتح الـ tunnel...')
tunnel_ready.wait(timeout=30)

if tunnel_url:
    ws_url = tunnel_url.replace('https://', 'wss://')
    print('\n' + '='*50)
    print('🌐 الـ URL بتاعك:')
    print(f'   {ws_url}')
    print('='*50)
    print('1. افتح الـ Dashboard في المتصفح')
    print('2. اضغط WS: OFF')
    print('3. الصق الـ URL وادغط Connect')
    print('='*50)
    print()
else:
    print('⚠️ Cloudflare tunnel تأخر — جرب تاني')

# تشغيل الـ bridge
async def run_server():
    server = bridge_mod.CinematicBridgeServer(engine=_engine)
    await server.start()

try:
    loop = asyncio.get_event_loop()
    loop.run_until_complete(run_server())
except KeyboardInterrupt:
    print('\n⏹ Bridge stopped')
